# S2 — **Schwarzschild–de Sitter** (GINN + Laplace)
$f=1-2M/r-\Lambda r^2/3$, with hair $\log_{10}(\Lambda/{\rm m^{-2}})\sim\mathcal U(-70,-20)$.
We use precession here. It helps putting an upper limit. 

In [ ]:
import sys, copy, warnings, dataclasses
sys.path.insert(0, '..')                      # the package lives one level up
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np, torch
import matplotlib.pyplot as plt
%matplotlib inline

from ginn.core import S2Data, keplerian_prefit, LossConfig, train, objective, fit
from ginn.physics import METRICS
from ginn import analysis as an
from ginn import plotting as pl

data = S2Data.from_csv('../data/tab_gillessen_pos.csv', '../data/tab_gillessen_vr.csv')
print(data.summary())

In [ ]:
N_STEPS_GINN   = 3000        
N_STEPS_KEPLER = 2000       
N_STEPS_EXACT  = 1500       
MODEL = dict(hidden=(32, 32),       
             nn_out_scale=1e-3)      
TRAIN = dict(lr_u=0.1,               
             lr_nn=5e-4,            
             lr_floor=0.01,          
             precondition=(0.0,))    

cfg = LossConfig(eps_phys=1e-4,      
                 n_colloc=256,       
                 sigma_w=10.0)       
FRAME = ['x0', 'y0', 'vx0', 'vy0', 'vz0']
LIT = {'inc': 134.18, 'Omega': 226.94, 'omega': 65.51}   

METRIC    = 'sds'
HAIR_INIT = {'log10_Lambda': -36.5}   # dont choose smaller than -34 - then the orbit is less bound
FIXED     = ()        
SCAN      = None   
N_STEPS_WARM = 1000        
HOLD_HAIR_IN_ADAM = False  
                           
                           
GN = dict(n_iter=30,      
          n_retrain=400,   
          tol=0.02,        
          lr_nn=2e-3)      
USE_FSP = False            # add precessionf_SP = 1.10 +- 0.19
CORNER = dict(n_samples=200_000,   
              bins=40,             # histogram bins per axis
              smooth=1.5)          # Gaussian smoothing of the 2-D histograms (in bins)
metric = METRICS[METRIC]
if USE_FSP:
    cfg = dataclasses.replace(cfg, external_precession=(1.10, 0.19))
HOLD = tuple(hp.name for hp in metric.hair if hp.name not in FIXED)
for hp in metric.hair: print(hp)

In [ ]:
# from ginn.physics import orbit_constants, M_SUN_GEOM_AU
# M_, a_, e_ = torch.tensor(4.26e6 * M_SUN_GEOM_AU), torch.tensor(1031.2), torch.tensor(0.8846)
# gr = 6 * np.pi * M_ / (a_ * (1 - e_**2))
# for hp in metric.hair:
#     for v in np.linspace(hp.prior.lo, hp.prior.hi, 5):
#         par = {h.name: torch.tensor(float(v if h.name == hp.name else HAIR_INIT.get(h.name, h.init))) for h in metric.hair}
#         d = orbit_constants(metric, a_, e_, par, M_)[3]; an_ = metric.precession(M_, a_, e_, par)
#         print(f"{hp.name} = {v:9.4g}: f_SP exact {(d/gr).item():9.4f}   analytic {(an_/gr).item():9.4f}")

## Pretraining

In [ ]:
init = keplerian_prefit(data)
ginn, hist = fit(metric, data, init, timing='ginn', cfg=cfg, n_steps=N_STEPS_GINN,
                 hair_init=HAIR_INIT, fixed=tuple(set(HOLD) | set(FIXED)) if HOLD_HAIR_IN_ADAM else FIXED,
                 **MODEL, **TRAIN)
if HOLD_HAIR_IN_ADAM:
    for h in HOLD:
        ginn.fix(h, fixed=False)                 # release the hair for the polish
print('after Adam:', ginn.summary())
pl.plot_loss_history(hist, f'GINN, {metric.label}: Adam stage'); plt.show()
pl.plot_param_history(hist, lit=LIT); plt.show()

## Gauss–Newton

In [ ]:
lap, gn_hist = an.gn_polish(ginn, data, cfg, **GN)
print('\nafter polish:', ginn.summary())
if HOLD:
    fig, ax = plt.subplots(1, 1 + len(HOLD), figsize=(6 * (1 + len(HOLD)), 3.5))
    ax[0].semilogy([max(h['chi2_gain'], 1e-4) for h in gn_hist], 'o-'); ax[0].axhline(GN['tol'], color='k', ls='--')
    ax[0].set_title('predicted chi2 gain per step (converged below dashed line)'); ax[0].set_xlabel('GN step')
    for a_, h in zip(ax[1:], HOLD):
        a_.plot([x[h] for x in gn_hist], 'o-'); a_.set_title(f'{h} during the polish'); a_.set_xlabel('GN step')
    plt.tight_layout(); plt.show()

## Results

In [ ]:
post = an.posterior(ginn, data, cfg)
est = an.print_estimates(ginn, post)
S = an.sample_laplace(ginn, post, n=CORNER['n_samples'])
print()
for k in ['M', 'D', 'a', 'e', 'inc', 'Omega', 'omega', 't0'] + FRAME:
    m_, lo_, hi_ = an.summarize(S)[k]; print(f"{k:6s} = {m_:.6g} +- {0.5 * (lo_ + hi_):.3g}")

In [ ]:
# save the best fit and its posterior (used by compare_orbits.ipynb and 00_validation_and_checks.ipynb)
import os
os.makedirs('results', exist_ok=True)
torch.save(dict(model=ginn, hair=est, post=post, cfg=cfg), f'results/{METRIC}.pt')
print(f'saved results/{METRIC}.pt')

## Upper limit on $log_{10}\Lambda$
A Gaussian (Laplace) cannot represent this posterior. The likelihood is flat below about −38 or so. Here the metric is indistinguishable from Schwarzschild, and it falls off at large $\Lambda$. We do the following:

1. At each grid value, re-converge the network model with the Gauss–Newton steps (network marginalised).
2. Weight each grid cell by its Laplace evidence, exp(−χ²/2)·√det Σ × prior × cell width. The lowest cell extends down to the prior edge −70.
3. Draw samples of all parameters from the mixture. The corner plot uses these samples.

The astrometry and velcoity data alone can mildly prefer more precession than GR, and a positive $\Lambda$ can source it. Without the GRAVITY precession data the posterior therefore shows a non-significant bump near 10⁻³⁵ m⁻². 
We print,

* $\Delta\chi^2$ = 2.71: one-sided 95%, independent of the prior.
* 95% credible bound: this depends on the −70 edge.

In [ ]:
GRID_POST = [-40.0, -37.5, -37.0, -36.75, -36.5, -36.25, -36.0, -35.75, -35.5, -35.0, -34.5]
MODE = 'lower_edge'
H = HOLD[0]
prof = an.gn_profile(ginn, data, cfg, H, GRID_POST, n_iter=4, n_retrain=GN['n_retrain'], lr_nn=GN['lr_nn'])
prior = metric.hair_by_name(H).prior
v = np.array(GRID_POST); mids = 0.5 * (v[1:] + v[:-1])
lo0 = prior.lo if MODE == 'lower_edge' else 0.0
edges = np.concatenate([[lo0], mids, [min(prior.hi, v[-1] + (v[-1] - mids[-1]))]])
entries = [dict(model=r['model'], lap=r['lap'], chi2=r['chi2'], cell=(H, lo, hi))
           for r, lo, hi in zip(prof['rows'], edges[:-1], edges[1:])]
S, _ = an.mixture_posterior(entries, n=CORNER['n_samples'])
if MODE == 'symmetric':
    S[H] = S[H] * np.random.default_rng(1).choice([-1.0, 1.0], size=len(S[H]))
print()
for r, d in zip(prof['rows'], prof['dchi2']):
    print(f"{H} = {r['value']:9.3f}   Delta chi2 = {d:8.3f}")
if prof['dchi2'][-1] < 9:
    print('WARNING: the last grid point is not strongly excluded (Delta chi2 < 9): extend the grid')
lab = metric.hair_by_name(H).latex
if MODE == 'lower_edge':
    up_cred = float(np.percentile(S[H], 95))
    print(f"\none-sided 95% upper limit (Delta chi2 = 2.71, prior-independent): {H} < {prof['upper_1sided95']:.2f}"
          f"   (Lambda < {10**prof['upper_1sided95']:.2e} m^-2)")
    print(f"95% credible upper bound with prior U({prior.lo:g}, {prior.hi:g}): {H} < {up_cred:.2f}  (prior-dependent)")
else:
    up_cred = float(np.percentile(np.abs(S[H]), 95))
    print(f"\n95% bound: |{H}| < {up_cred:.3g}   (Delta chi2 = 2.71 on |{H}|: {prof['upper_1sided95']:.3g})")
    print(f"posterior: {H} = {np.median(S[H]):.3g} +- {0.5*(np.percentile(S[H],84.135)-np.percentile(S[H],15.865)):.3g}")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(S[H], bins=150, density=True, color='#1f4e9c', histtype='stepfilled', alpha=0.6)
for sgn in ([1] if MODE == 'lower_edge' else [-1, 1]):
    ax[0].axvline(sgn * up_cred, color='k', ls=':', label='95% bound' if sgn == 1 else None)
ax[0].set_xlabel(lab); ax[0].set_ylabel('posterior'); ax[0].legend()
ax[1].plot(prof['values'], prof['dchi2'], 'o-', color='#1f4e9c'); ax[1].axhline(2.71, color='r', ls='--')
ax[1].set_xlabel(lab if MODE == 'lower_edge' else '|' + H + '|'); ax[1].set_ylabel(r'$\Delta\chi^2$')
ax[1].set_ylim(-0.3, min(40, max(10, prof['dchi2'].max()))); plt.tight_layout(); plt.show()

In [ ]:
r_ = torch.load(f'results/{METRIC}.pt', weights_only=False)
r_['profile'] = dict(values=prof['values'], chi2=prof['chi2'])
ref_file = 'results/schwarzschild.pt'
ref = torch.load(ref_file, weights_only=False) if os.path.exists(ref_file) else {}
if 'evidence' in ref:
    r_['bayes'] = an.bayes_factor_profile(prof, prior, ref['evidence'], lower_edge=(MODE == 'lower_edge'))
    an.print_bayes(metric.label, r_['bayes'])
else:
    print('run 01_schwarzschild.ipynb first: it provides the reference evidence')
torch.save(r_, f'results/{METRIC}.pt')

## Corner plots

In [ ]:
# one corner plot with ALL fitted parameters: orbit, hair and reference-frame offsets
keys = ['M', 'D', 'a', 'e', 'inc', 'Omega', 'omega', 't0'] + FRAME + list(HOLD)   # ALL parameters, one plot
pl.corner_plot([S], an.labels_for(ginn), keys, names=['GINN + Laplace'],
               bins=CORNER['bins'], smooth=CORNER['smooth'],
               title=f'S2, {metric.label}: 1σ / 2σ contours (orbit, frame, hair)')
plt.savefig("corner_04_sds.png")
plt.show()

In [ ]:
keys_red = ['M', 'a', 'e', 'inc', 'Omega', 'omega', 't0'] + list(HOLD)
RANGES = {'log10_Lambda': (-70, -25)} 
pl.corner_plot([S], an.labels_for(ginn), keys_red, names=['GINN + Laplace'],
               bins=CORNER['bins'], smooth=CORNER['smooth'],
               title=f'S2, {metric.label}: mass, orbit and hair')
plt.savefig("small_corner_04_sds.png")
plt.show()

In [ ]:
import corner
from matplotlib.colors import to_rgba

keys = ['M', 'D', 'a', 'e', 'inc', 'Omega', 'omega', 't0']+ list(HOLD)   # reduced plot: ['M', 'a', 'e', 'inc', 'Omega', 'omega', 't0'] + list(HOLD)
RANGES = {'log10_Lambda': (-70, -25)}       # e.g. (-45, -30) to zoom in on the drop
labels = an.labels_for(ginn)
c = '#1f4e9c'

X = np.column_stack([np.asarray(S[k]) for k in keys])
fig = corner.corner(
    X, labels=[labels[k] for k in keys], range=[RANGES.get(k, 1.0) for k in keys],
    bins=CORNER['bins'], smooth=CORNER['smooth'], smooth1d=CORNER['smooth'], color=c,
    levels=(1 - np.exp(-0.5), 1 - np.exp(-2.0)),                       # 1 sigma and 2 sigma in 2D
    plot_datapoints=False, plot_density=False, fill_contours=True, no_fill_contours=True,
    contourf_kwargs=dict(colors=[(1, 1, 1, 0), to_rgba(c, 0.18), to_rgba(c, 0.45)]),
    contour_kwargs=dict(colors=[c], linewidths=1.2),
    quantiles=[0.15865, 0.5, 0.84135], label_kwargs=dict(fontsize=12),
    hist_kwargs=dict(color=c, lw=1.4), max_n_ticks=3, labelpad=0.08)

K = len(keys)
axes = np.array(fig.axes).reshape(K, K)
for i, k in enumerate(keys):
    if k == 'log10_Lambda':                  # one-sided posterior: quote the upper limit
        axes[i, i].set_title(f"{labels[k]} < {np.percentile(S[k], 95):.2f} (95%)", fontsize=10)
        continue
    lo, med, hi = np.percentile(S[k], [15.865, 50, 84.135])
    err = min(med - lo, hi - med)
    dec = int(max(2, 1 - np.floor(np.log10(err)))) if err > 0 else 2
    axes[i, i].set_title(f"{labels[k]} = ${med:.{dec}f}^{{+{hi - med:.{dec}f}}}_{{-{med - lo:.{dec}f}}}$",
                         fontsize=10)
for ax in fig.axes:                          # plain tick labels (2002.32, not +2.0023e3)
    for axis in (ax.xaxis, ax.yaxis):
        f = axis.get_major_formatter()
        if hasattr(f, 'set_useOffset'):
            f.set_useOffset(False)
fig.suptitle(f'S2, {metric.label}: 1σ / 2σ contours', fontsize=14)
plt.savefig("small_corner_04_sds.png")
plt.show()

In [ ]:
import corner
from matplotlib.colors import to_rgba

keys = ['M', 'D', 'a', 'e', 'inc', 'Omega', 'omega', 't0'] + FRAME + list(HOLD)   # reduced plot: ['M', 'a', 'e', 'inc', 'Omega', 'omega', 't0'] + list(HOLD)
RANGES = {'log10_Lambda': (-70, -25)}       # e.g. (-45, -30) to zoom in on the drop
labels = an.labels_for(ginn)
c = '#1f4e9c'

X = np.column_stack([np.asarray(S[k]) for k in keys])
fig = corner.corner(
    X, labels=[labels[k] for k in keys], range=[RANGES.get(k, 1.0) for k in keys],
    bins=CORNER['bins'], smooth=CORNER['smooth'], smooth1d=CORNER['smooth'], color=c,
    levels=(1 - np.exp(-0.5), 1 - np.exp(-2.0)),                       # 1 sigma and 2 sigma in 2D
    plot_datapoints=False, plot_density=False, fill_contours=True, no_fill_contours=True,
    contourf_kwargs=dict(colors=[(1, 1, 1, 0), to_rgba(c, 0.18), to_rgba(c, 0.45)]),
    contour_kwargs=dict(colors=[c], linewidths=1.2),
    quantiles=[0.15865, 0.5, 0.84135], label_kwargs=dict(fontsize=12),
    hist_kwargs=dict(color=c, lw=1.4), max_n_ticks=3, labelpad=0.08)

K = len(keys)
axes = np.array(fig.axes).reshape(K, K)
for i, k in enumerate(keys):
    if k == 'log10_Lambda':                  # one-sided posterior: quote the upper limit
        axes[i, i].set_title(f"{labels[k]} < {np.percentile(S[k], 95):.2f} (95%)", fontsize=10)
        continue
    lo, med, hi = np.percentile(S[k], [15.865, 50, 84.135])
    err = min(med - lo, hi - med)
    dec = int(max(2, 1 - np.floor(np.log10(err)))) if err > 0 else 2
    axes[i, i].set_title(f"{labels[k]} = ${med:.{dec}f}^{{+{hi - med:.{dec}f}}}_{{-{med - lo:.{dec}f}}}$",
                         fontsize=10)
for ax in fig.axes:                          # plain tick labels (2002.32, not +2.0023e3)
    for axis in (ax.xaxis, ax.yaxis):
        f = axis.get_major_formatter()
        if hasattr(f, 'set_useOffset'):
            f.set_useOffset(False)
fig.suptitle(f'S2, {metric.label}: 1σ / 2σ contours', fontsize=14)
plt.show()

In [ ]:
#pl.plot_convergence(hist, S, gn_hist if HOLD else None, title=f'{metric.label}: convergence'); plt.show()

In [ ]:
# keep the posterior samples for the summary plots in 09_compare_orbits
import os
res_file = f'results/{METRIC}.pt'
if os.path.exists(res_file):
    r_ = torch.load(res_file, weights_only=False)
    r_['samples'] = {k: np.asarray(v)[:20000] for k, v in S.items()}
    torch.save(r_, res_file)

In [ ]:
# 1-D posterior of log10(Lambda) over the full prior range, from the section-3b samples S
H = 'log10_Lambda'
prior = metric.hair_by_name(H).prior                 # U(-70, -20)
x = np.sort(S[H])
assert x.min() < prior.lo + 5, "S must be the section-3b (one-sided) samples - re-run section 3b first"

# cumulative probability: the fraction of samples below each value
cdf = np.arange(1, x.size + 1) / x.size
levels = [0.68, 0.95]                                # use 0.65 here if you really want 65%
upper = {p: float(np.interp(p, cdf, x)) for p in levels}

bins = np.linspace(prior.lo, prior.hi, 201)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
ax[0].hist(x, bins=bins, density=True, histtype='stepfilled', color='#1f4e9c', alpha=0.5)
ax[0].hist(x, bins=bins, density=True, histtype='step', color='#1f4e9c', lw=1.2)
ax[1].plot(x, cdf, color='#1f4e9c', lw=1.5)
for p, ls in zip(levels, [':', '--']):
    lab = f'{int(round(100 * p))}% upper limit: {upper[p]:.2f}'
    ax[0].axvline(upper[p], color='k', ls=ls, label=lab)
    ax[1].axvline(upper[p], color='k', ls=ls, label=lab)
    ax[1].axhline(p, color='0.6', ls=ls, lw=0.8)
ax[0].set_ylabel('posterior density')
ax[1].set_ylabel('cumulative probability')
for a in ax:
    a.set_xlim(prior.lo, prior.hi)
    a.set_xlabel(r'$\log_{10}(\Lambda/{\rm m^{-2}})$')
    a.legend(fontsize=9, loc='upper left')
plt.tight_layout(); plt.show()

for p in levels:
    print(f"{int(round(100 * p))}% credible upper limit: log10(Lambda/m^-2) < {upper[p]:.2f}"
          f"   ->   Lambda < {10 ** upper[p]:.2e} m^-2")
print(f"(prior-independent Delta chi2 = 2.71 limit from section 3b: log10 Lambda < {prof['upper_1sided95']:.2f})")

## Some validation

In [ ]:
rep = an.nn_role_report(ginn, data, cfg, lap=post)
pl.plot_timing(rep['timing_curves'], data); plt.show()
pl.plot_fit(ginn, data); plt.show()

In [ ]:
# exact, _ = fit(metric, data, init, timing='exact', cfg=cfg, n_steps=N_STEPS_EXACT,
#                hair_init=HAIR_INIT, fixed=FIXED, **TRAIN)
# lap_ex = an.laplace(exact, data, cfg)
# print('exact solver:', exact.summary())
# print('exact solver hair:', an.hair_estimates(exact, lap_ex))

## Multi-start consistency test
The full procedure (Adam, then Gauss–Newton polish, then Laplace) is run again from several starting values of the hair parameter. Schwarzschild has no hair, so the network's random seed is varied instead. The result after Adam and after the polish is printed for each start. The estimate is accepted only if the spread of the final values is **< 0.2σ**.
**SdS:** below log₁₀Λ ≈ −38 every value fits equally well (the likelihood is flat), so starts in that region need not agree. The consistent quantity for SdS is the upper limit in section 3b.

In [ ]:
RUN_MULTISTART = True
MULTI_STARTS = [-45.0, -38.0, -34.0]
ms_rows = []
if RUN_MULTISTART:
    for s0 in MULTI_STARTS:
        if HOLD:
            hi, seed = {**HAIR_INIT, HOLD[0]: s0}, 0
        else:
            hi, seed = HAIR_INIT, int(s0)
        m_, h_ = fit(metric, data, init, timing='ginn', cfg=cfg, n_steps=N_STEPS_GINN, hair_init=hi,
                     fixed=FIXED, seed=seed, progress=False, **MODEL, **TRAIN)
        adam = {k: float(m_.physical()['par'][k]) for k in HOLD}
        chi2_adam = an.chi2_parts(m_, data)['total']
        lap_, gh_ = an.gn_polish(m_, data, cfg, verbose=False, **GN)
        post_ = an.posterior(m_, data, cfg, verbose=False)
        S_ = an.sample_laplace(m_, post_, n=20000, verbose=False)
        row = dict(start=s0, adam=adam, chi2_adam=chi2_adam, est=an.hair_estimates(m_, post_),
                   chi2=an.chi2_parts(m_, data)['total'], steps=len(gh_),
                   M=an.summarize(S_)['M'], D=an.summarize(S_)['D'])
        ms_rows.append(row)
        txt = '  '.join(f"{k}: Adam {row['adam'][k]:+.5g} -> polish {row['est'][k][0]:+.5g} +- {row['est'][k][1]:.3g}" for k in HOLD)
        print(f"start {s0:+g}: {txt}  | M = {row['M'][0]:.4f}, R0 = {row['D'][0]:.4f} | chi2 Adam {chi2_adam:.2f} -> {row['chi2']:.2f} ({row['steps']} GN steps)")
    def spread(vals, sig):
        return (max(vals) - min(vals)) / sig
    sym = lambda k, v: abs(v) if k in an.SQUARED_HAIR else v
    checks = [(k, [sym(k, r['est'][k][0]) for r in ms_rows], np.mean([r['est'][k][1] for r in ms_rows])) for k in HOLD]
    checks += [('M', [r['M'][0] for r in ms_rows], np.mean([0.5 * (r['M'][1] + r['M'][2]) for r in ms_rows])),
               ('R0', [r['D'][0] for r in ms_rows], np.mean([0.5 * (r['D'][1] + r['D'][2]) for r in ms_rows]))]
    print()
    for k, vals, sig in checks:
        sp = spread(vals, sig)
        print(f"{k:>14s}: spread across starts = {sp:.2f} sigma  ->  " +
              ('CONSISTENT' if sp < 0.2 else 'NOT CONSISTENT (increase N_STEPS_GINN or GN n_iter)'))
    if HOLD:
        k = HOLD[0]
        fig, ax = plt.subplots(figsize=(7, 4))
        xs = np.arange(len(ms_rows))
        ax.errorbar(xs, [r['est'][k][0] for r in ms_rows], [r['est'][k][1] for r in ms_rows], fmt='o',
                    color='#1f4e9c', capsize=4, label='after polish (± Laplace)')
        ax.plot(xs, [r['adam'][k] for r in ms_rows], 'x', color='0.4', label='after Adam')
        ax.plot(xs, [r['start'] for r in ms_rows], '.', color='#d1495b', label='start value')
        ax.set_xticks(xs); ax.set_xticklabels([f"start {r['start']:+g}" for r in ms_rows])
        ax.set_ylabel(k); ax.legend(fontsize=9); ax.set_title('multi-start consistency'); plt.show()